# Data Prep for Neural Network

In [1]:
from datetime import datetime

import polars as pl

from run_config import (
    PATHS,
    RUN_MODE,
    MODEL_START_DATE,
    MODEL_END_DATE,
)

## Train Test Split

The input and output paths are selected by `RUN_MODE` in `run_config.py`. Thus, both `sample` and `full` mode write to separate locations. The data can be split randomly (70/15/15) or chronologically.

In [2]:
DATASETS = (
    PATHS.gold_1h_demand_hexagon,
    PATHS.gold_1h_demand_census_tracts,
    PATHS.gold_1h_demand_community_areas,
    PATHS.gold_2h_demand_hexagon,
    PATHS.gold_2h_demand_census_tracts,
    PATHS.gold_2h_demand_community_areas,
    PATHS.gold_4h_demand_hexagon,
    PATHS.gold_4h_demand_census_tracts,
    PATHS.gold_4h_demand_community_areas,
)
OUTPUT_DIR = PATHS.train_test_dir
TARGET_COL = "trip_count"

SEED = 42
RANDOM = True

MODEL_START_TS = datetime.fromisoformat(MODEL_START_DATE)
MODEL_END_TS = datetime.fromisoformat(MODEL_END_DATE)
if MODEL_START_TS >= MODEL_END_TS:
    raise ValueError(
        f"MODEL_START_DATE must be before MODEL_END_DATE: "
        f"{MODEL_START_DATE} >= {MODEL_END_DATE}"
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Run mode: {RUN_MODE}")
if RUN_MODE == "full":
    print(f"Model period: [{MODEL_START_DATE}, {MODEL_END_DATE})")
else:
    print("Model-period filter disabled in sample mode")
print(f"Inputs: {[path.name for path in DATASETS]}")
print(f"Output directory: {OUTPUT_DIR}")

Run mode: full
Model period: [2025-01-01T00:00:00, 2026-05-01T00:00:00)
Inputs: ['GOLD_1H_DEMAND_HEXAGON.parquet', 'GOLD_1H_DEMAND_CENSUS_TRACTS.parquet', 'GOLD_1H_DEMAND_COMMUNITY_AREAS.parquet', 'GOLD_2H_DEMAND_HEXAGON.parquet', 'GOLD_2H_DEMAND_CENSUS_TRACTS.parquet', 'GOLD_2H_DEMAND_COMMUNITY_AREAS.parquet', 'GOLD_4H_DEMAND_HEXAGON.parquet', 'GOLD_4H_DEMAND_CENSUS_TRACTS.parquet', 'GOLD_4H_DEMAND_COMMUNITY_AREAS.parquet']
Output directory: /Users/lennartjekel/dev/git/Group-3-AAA/data/full/train_test_data


In [3]:
def create_splits(dataset_path):
    df_split = pl.scan_parquet(dataset_path)

    if RUN_MODE == "full":
        df_split = df_split.filter(
            (pl.col("datetime_hour") >= MODEL_START_TS)
            & (pl.col("datetime_hour") < MODEL_END_TS)
        )

    if RANDOM:
        bucketed = (
            df_split
            .with_row_index("_row_id")
            .with_columns(
                (pl.col("_row_id").hash(seed=SEED) % 100).alias("_split_bucket")
            )
        )
        train = bucketed.filter(pl.col("_split_bucket") < 70)
        val = bucketed.filter(
            (pl.col("_split_bucket") >= 70) & (pl.col("_split_bucket") < 85)
        )
        test = bucketed.filter(pl.col("_split_bucket") >= 85)
        helper_columns = ["_row_id", "_split_bucket"]
        train = train.drop(helper_columns)
        val = val.drop(helper_columns)
        test = test.drop(helper_columns)
    else:
        train = df_split.filter(pl.col("datetime_hour") < pl.datetime(2025, 9, 1))
        val = df_split.filter(
            (pl.col("datetime_hour") >= pl.datetime(2025, 9, 1))
            & (pl.col("datetime_hour") < pl.datetime(2026, 1, 1))
        )
        test = df_split.filter(pl.col("datetime_hour") >= pl.datetime(2026, 1, 1))

    return df_split, train, val, test


split_results = {}
for dataset_path in DATASETS:
    df_split, train, val, test = create_splits(dataset_path)
    output_paths = {
        "train": OUTPUT_DIR / f"{dataset_path.stem}_TRAIN.parquet",
        "val": OUTPUT_DIR / f"{dataset_path.stem}_VAL.parquet",
        "test": OUTPUT_DIR / f"{dataset_path.stem}_TEST.parquet",
    }

    counts = {
        "total": df_split.select(pl.len()).collect().item(),
        "train": train.select(pl.len()).collect().item(),
        "val": val.select(pl.len()).collect().item(),
        "test": test.select(pl.len()).collect().item(),
    }
    if counts["total"] == 0:
        raise ValueError(f"Input dataset is empty: {dataset_path}")
    if counts["train"] + counts["val"] + counts["test"] != counts["total"]:
        raise ValueError(f"Split counts do not add up for {dataset_path}")

    train.sink_parquet(output_paths["train"])
    val.sink_parquet(output_paths["val"])
    test.sink_parquet(output_paths["test"])
    split_results[dataset_path.stem] = {"counts": counts, "paths": output_paths}

    shares = {name: round(count / counts["total"], 2) for name, count in counts.items() if name != "total"}
    print(f"{dataset_path.name}: {counts}, shares={shares}")
    print(f"Written: {output_paths}")

GOLD_1H_DEMAND_HEXAGON.parquet: {'total': 9928920, 'train': 6948550, 'val': 1487826, 'test': 1492544}, shares={'train': 0.7, 'val': 0.15, 'test': 0.15}
Written: {'train': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/full/train_test_data/GOLD_1H_DEMAND_HEXAGON_TRAIN.parquet'), 'val': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/full/train_test_data/GOLD_1H_DEMAND_HEXAGON_VAL.parquet'), 'test': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/full/train_test_data/GOLD_1H_DEMAND_HEXAGON_TEST.parquet')}
GOLD_1H_DEMAND_CENSUS_TRACTS.parquet: {'total': 10219920, 'train': 7152474, 'val': 1531283, 'test': 1536163}, shares={'train': 0.7, 'val': 0.15, 'test': 0.15}
Written: {'train': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/full/train_test_data/GOLD_1H_DEMAND_CENSUS_TRACTS_TRAIN.parquet'), 'val': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/full/train_test_data/GOLD_1H_DEMAND_CENSUS_TRACTS_VAL.parquet'), 'test': PosixPath('/Users/lennartjeke

In [4]:
df_split.head(10).collect()

datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,tmpc,relh,sknt,vsby,p01m,skyc1_BKN,skyc1_CLR,skyc1_FEW,skyc1_OVC,skyc1_SCT,skyc1_VV,date,is_holiday,community_area,food_drink,landmark,shop,train_station,trip_count,trip_seconds_sum,trip_seconds_mean,trip_seconds_min,trip_seconds_max,trip_miles_sum,trip_miles_mean,trip_miles_min,trip_miles_max,fare_sum,fare_mean,fare_min,fare_max,tips_sum,tips_mean,tips_min,tips_max,tolls_sum,tolls_mean,tolls_min,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
datetime[μs],i8,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,i8,i8,i8,i8,date,i8,i64,f64,f64,f64,f64,u32,i64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
2025-06-12 12:00:00,6,4,12,0.5,-0.866025,0.433884,-0.900969,1.2246e-16,-1.0,25.0,51.17,3.25,9.25,0.0,0,0,1,0,0,0,2025-06-12,0,50,11.0,4.0,6.0,0.0,10,14890,1489.0,999,2340,120.64,12.064,10.35,14.3,317.0,31.7,28.5,39.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,317.0,31.7,28.5,39.25,"""Prcard"""
2025-05-11 16:00:00,5,7,16,0.866025,-0.5,-0.781831,0.62349,-0.866025,-0.5,20.695,38.8425,8.5,10.0,0.0,0,0,1,0,0,0,2025-05-11,0,50,11.0,4.0,6.0,0.0,8,12947,1618.375,1007,2802,89.21,11.15125,6.1,21.69,249.6,31.2,18.5,61.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.5,0.3125,0.0,1.5,252.1,31.5125,18.5,61.1,"""Prcard"""
2025-05-26 20:00:00,5,1,20,0.866025,-0.5,0.0,1.0,-0.866025,0.5,17.915,41.7925,12.25,10.0,0.0,1,0,0,0,0,0,2025-05-26,1,50,11.0,4.0,6.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
2025-03-28 20:00:00,3,5,20,0.866025,0.5,-0.433884,-0.900969,-0.866025,0.5,25.9725,32.6975,20.75,10.0,0.0,0,0,0,0,1,0,2025-03-28,0,50,11.0,4.0,6.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
2025-06-12 08:00:00,6,4,8,0.5,-0.866025,0.433884,-0.900969,0.866025,-0.5,21.2525,59.5875,4.0,9.5,0.0,0,0,1,0,0,0,2025-06-12,0,50,11.0,4.0,6.0,0.0,7,9693,1384.714286,852,1837,65.39,9.341429,7.44,10.96,180.07,25.724286,20.25,28.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.142857,0.0,1.0,181.07,25.867143,20.25,28.75,"""Prcard"""
2025-04-06 20:00:00,4,7,20,1.0,6.1232e-17,-0.781831,0.62349,-0.866025,0.5,7.915,32.0575,6.5,10.0,0.0,0,0,1,0,0,0,2025-04-06,0,50,11.0,4.0,6.0,0.0,6,6613,1102.166667,840,1609,69.06,11.51,9.1,13.77,179.0,29.833333,24.0,34.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,179.0,29.833333,24.0,34.25,"""Prcard"""
2025-05-02 16:00:00,5,5,16,0.866025,-0.5,-0.433884,-0.900969,-0.866025,-0.5,11.668,66.956,5.8,10.0,0.0,0,0,0,1,0,0,2025-05-02,0,50,11.0,4.0,6.0,0.0,8,11366,1420.75,378,3900,64.33,8.04125,0.36,18.84,199.42,24.9275,6.0,49.17,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.125,0.0,1.0,200.92,25.115,6.0,49.17,"""Prcard"""
2025-03-13 12:00:00,3,4,12,0.866025,0.5,0.433884,-0.900969,1.2246e-16,-1.0,8.1925,52.7225,5.0,10.0,0.0,0,0,1,0,0,0,2025-03-13,0,50,11.0,4.0,6.0,0.0,10,10708,1070.8,786,1330,96.01,9.601,3.63,13.4,257.24,25.724,14.24,33.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.1,0.0,1.0,258.74,25.874,14.74,33.5,"""Prcard"""
2025-04-08 00:00:00,4,2,0,1.0,6.1232e-17,0.781831,0.62349,0.0,1.0,0.90375,57.295,10.25,10.0,0.0006,1,0,0,0,0,0,2025-04-08,0,50,11.0,4.0,6.0,0.0,2,1862,931.0,660,1202,22.06,11.03,10.26,11.8,56.75,28.375,27.25,29.5,0.09,0.045,0.0,0.09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,56.84,28.42,27.34,29.5,"""Prcard"""
